# Multi-Layer Automotive Intrusion Detection System (IDS)
### Complete Implementation with Performance Visualizations

This notebook demonstrates the end-to-end pipeline for an automotive IDS using the **consolidated HCRL dataset** (DoS, Fuzzy, Spoofing, and Normal traffic).

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Add src to path
sys.path.append(os.getcwd())

from src.parser import CANParser
from src.features import CANFeatureEngineer
from src.validation import DataSplitter
from src.models import CANModelTrainer
from src.evaluation import IDSEvaluator
from src.telematics import TelematicsProcessor, BehavioralBaseline, AutoencoderBaseline
from src.integration import AlertCorrelator
from src.plotting import IDSPlotter

print("Setup Complete. All Pipeline Components Loaded.")

## 1. Data Exploration & Multi-Attack Analysis
We load the consolidated dataset and visualize patterns across different attack types.

In [ ]:
# Load and Preprocess Consolidated Dataset
consolidated_file = 'Data/consolidated_dataset.csv'
df = next(CANParser.load_consolidated(consolidated_file, chunksize=100000))
df = CANParser.preprocess_df(df)
df = CANFeatureEngineer.extract_message_features(df)
df = CANFeatureEngineer.extract_window_features(df, window_size=50)

print(f"Loaded consolidated dataset with attack types: {df['attack_type'].unique()}")

# Visualizing Attack Patterns (DoS section)
dos_sample = df[df['attack_type'] == 'DoS'].iloc[100:1100]
IDSPlotter.plot_attack_patterns(dos_sample, title="Consolidated Data: DoS Attack Manifestation")

## 2. Multi-Class Model Training & Evaluation
Training an **XGBoost Classifier** to detect and classify various attack types.

In [ ]:
X, y = CANFeatureEngineer.get_feature_matrix(df)
X_train, X_test, y_train, y_test = DataSplitter.temporal_split(X, y, test_size=0.2)

trainer = CANModelTrainer(model_type='xgboost')
trainer.train(X_train, y_train)
trainer.save_model('models/final_xgb.joblib')

# Predictions for Evaluation
y_pred = trainer.predict(X_test)
y_probs = trainer.model.predict_proba(X_test)[:, 1]

# Performance Visuals
IDSPlotter.plot_confusion_matrix(y_test, y_pred, title="XGBoost IDS: Multi-Attack Confusion Matrix")
IDSPlotter.plot_roc_curve(y_test, y_probs, title="XGBoost IDS: Multi-Attack ROC Curve")

## 3. Telematics Behavioral Monitoring (Dual-Model Approach)
Using the 'Normal' subset to establish both Statistical (3-Sigma) and Neural (Autoencoder) baselines.

In [ ]:
norm_df = df[df['attack_type'] == 'Normal'].copy()
tele_df = TelematicsProcessor.derive_features(norm_df)

# 1. Statistical Baseline (3-Sigma)
baseline = BehavioralBaseline()
baseline.fit(tele_df)
anomalies_sigma = baseline.detect_anomalies(tele_df, threshold_sigma=3)

# 2. Advanced Neural Baseline (Autoencoder)
ae_baseline = AutoencoderBaseline(hidden_layer_sizes=(8, 4, 8))
ae_baseline.fit(tele_df)
anomalies_ae = ae_baseline.detect_anomalies(tele_df)
ae_baseline.save_model('models/behavioral_ae.joblib')

print(f"Anomalies Detected (3-Sigma): {sum(anomalies_sigma)}")
print(f"Anomalies Detected (Autoencoder): {sum(anomalies_ae)}")

# Behavioral Visualization
IDSPlotter.plot_behavioral_envelope(tele_df, anomalies_ae, col='Speed', title="Telematics Baseline: Neural (Autoencoder) Speed Envelope")

## 4. Integrated Multi-Layer Alert Analysis
Mapping detections of specific attack types into vSOC alerts.

In [ ]:
# Example: Correlating a Spoofing detection with a behavioral anomaly
can_alerts = pd.DataFrame({'Timestamp': [df['Timestamp'].iloc[-1]], 'Attack_Type': ['Spoofing']})
tele_anomalies = pd.DataFrame({'Timestamp': [df['Timestamp'].iloc[-1]], 'Anomaly_Score': [1]})

correlated = AlertCorrelator.correlate(can_alerts, tele_anomalies)
vsoc_alert = AlertCorrelator.generate_vsoc_alert(correlated.iloc[0])

print("Final Correlated vSOC Alert:")
print(json.dumps(vsoc_alert, indent=2))